In [2]:
import pandas as pd
import requests
import time

In [8]:
url = "https://fapi.binance.com/fapi/v1/fundingRate"

start_time = int(pd.Timestamp("2023-05-07").timestamp() * 1000)
end_time = int(pd.Timestamp("2026-05-07").timestamp()*1000)

all_data = []
current_start = start_time

while current_start < end_time:
    params = {
        "symbol": "BTCUSDT",
        "startTime": current_start,
        "endTime": end_time,
        "limit": 1000
    }

    response = requests.get(url, params=params)
    data = response.json()

    if not data:
        break
    
    all_data.extend(data)
    
    #dictionary access
    last_time = data[-1]["fundingTime"]

    #move window forward +1 after last time
    current_start = last_time +1

    print(f"pulled up to: {pd.to_datetime(last_time, unit='ms')}")
    
    time.sleep(0.1)

#convert time stamps from ms
funding_df = pd.DataFrame(all_data)


pulled up to: 2024-04-04 00:00:00
pulled up to: 2025-03-03 08:00:00
pulled up to: 2026-01-30 16:00:00.001000
pulled up to: 2026-05-07 00:00:00


In [9]:
funding_df.head()

,symbol,fundingTime,fundingRate,markPrice
0,BTCUSDT,1683417600000,0.00004321,
1,BTCUSDT,1683446400000,0.00004047,
2,BTCUSDT,1683475200006,0.00006046,
3,BTCUSDT,1683504000000,0.00000720,
4,BTCUSDT,1683532800007,0.00000407,


### Cleaning funding rate

In [10]:
#drops markprice column
funding_df = funding_df.drop(columns=["markPrice"])
#changes name of column
funding_df = funding_df.rename(columns={"fundingRate": "funding_rate"})

#changes to datetime and ms
funding_df["fundingTime"] = pd.to_datetime(
    funding_df["fundingTime"],
    unit = "ms"
)

#changes the dtype
funding_df["date"] = (
    funding_df["fundingTime"].dt.normalize()
)

#changes funding rate from string to float
funding_df["funding_rate"] = funding_df["funding_rate"].astype(float)

#makes a new clean dataframe
daily_funding = (
    funding_df
    .groupby("date")["funding_rate"]
    .sum()
    .reset_index()
)

daily_funding.head()

,date,funding_rate
0,2023-05-07,0.000144
1,2023-05-08,0.000050
2,2023-05-09,0.000123
3,2023-05-10,0.000123
4,2023-05-11,0.000137


In [12]:
funding_df.to_csv("fundingrate_unmerged.csv")

In [13]:
daily_funding.to_csv("fundingratedata_mergeready.csv")

### Cleaning spot & future data

In [15]:
df = pd.read_csv("/Users/emmanuelarowolo/btc_spot_perp_meanreversion-1/data/processed/btcspot_and_future.csv")

## Next steps

Clean the df dataframe. With datetime  
Clean the funding df  
Add funding df to df cleanly by date and averaging funding rate over a day  
Then save this brand new dataframe as a csv  
Then continue on ou_model.ipynb  

In [16]:
df.head()
df["timestamps"] = pd.to_datetime(
    df["timestamp"],
    unit="ms"
)

In [17]:
df.head()

,timestamp,spot_price,future_price,timestamps
0,1683331200000,28848.20,28837.8,2023-05-06
1,1683417600000,28430.10,28419.4,2023-05-07
2,1683504000000,27668.79,27659.8,2023-05-08
3,1683590400000,27628.27,27610.2,2023-05-09
4,1683676800000,27598.75,27582.9,2023-05-10


In [18]:
df = df.drop(columns=["timestamp"])

In [19]:
df.head()

,spot_price,future_price,timestamps
0,28848.20,28837.8,2023-05-06
1,28430.10,28419.4,2023-05-07
2,27668.79,27659.8,2023-05-08
3,27628.27,27610.2,2023-05-09
4,27598.75,27582.9,2023-05-10


In [20]:
col = df.pop("timestamps")
df.insert(0, "timestamp", col)

df.head()

,timestamp,spot_price,future_price
0,2023-05-06,28848.20,28837.8
1,2023-05-07,28430.10,28419.4
2,2023-05-08,27668.79,27659.8
3,2023-05-09,27628.27,27610.2
4,2023-05-10,27598.75,27582.9


In [21]:
df.to_csv("timestamped_btcspot_perp.csv")

### Merging the two dataframes

In [23]:
#creating a date column for merging
df["date"] = df["timestamp"].dt.normalize()

In [26]:
#merging on head
merged_df = df.merge(
    daily_funding,
    on="date",
    how="left"
)

merged_df.head()

,timestamp,spot_price,future_price,date,funding_rate
0,2023-05-06,28848.20,28837.8,2023-05-06,NaN
1,2023-05-07,28430.10,28419.4,2023-05-07,0.000144
2,2023-05-08,27668.79,27659.8,2023-05-08,0.000050
3,2023-05-09,27628.27,27610.2,2023-05-09,0.000123
4,2023-05-10,27598.75,27582.9,2023-05-10,0.000123


In [ ]:
#drops the merge index date because it is the same as timestamp
merged_df.drop(
    columns=["date"],
    inplace=True
)

In [ ]:
#makes timestamp the index
merged_df.set_index(
    "timestamp",
    inplace=True
)

In [30]:
merged_df.to_csv("mergedbtc_and_fundingrate.csv")